# 02 · Train the hazard U-Net (Colab, GPU)

Runtime → **T4 GPU**. Run `01_render_dataset` first; this reads the dataset it wrote to Google Drive.

Trains on synthetic terrains, validates on held-out synthetic terrains, and reports the headline number on the **real Nightingale surface** (never trained on): per-class IoU, the false-safe rate (hazard called safe), and boulder recall by physical size.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /content/touch_down || (cd /content/touch_down && git pull -q)
%cd /content/touch_down
!pip install -q -e .
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/touchdown/dataset'
RUN = '/content/drive/MyDrive/touchdown/runs/unet_v1'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Copy the images to local disk first: reading thousands of small files from Drive is slow.
!mkdir -p /content/ds && cp -r {DATA}/images {DATA}/labels {DATA}/meta.jsonl /content/ds/

In [ ]:
!python -m touchdown.vision.train --data /content/ds --out {RUN} --epochs 30 --bs 8

In [ ]:
import json
r = json.load(open(f'{RUN}/results.json'))
for split, res in r.items():
    print(split, 'mIoU', round(res['miou'], 3), '| false-safe', round(res['false_safe_rate'], 3), '| false-hazard', round(res['false_hazard_rate'], 3))
    for b in res['boulder_recall_by_size']:
        print(f"   boulders {b['lo']}-{b['hi']} m: n={b['n']}, recall={b.get('recall', float('nan')):.2f}")